# Cost Telemetry
# 0. 介绍

**研究背景**：Agent 完成一项任务时，可能会连续调用多次大模型，并穿插路由、重试和缓存。Provider 只会为单次 API 请求返回 Token 用量，外层程序还必须把这些用量与具体任务、步骤和模型关联起来，才能知道钱花在哪里、是否超出预算。

**现存问题**：生产中常见的错误基线是只保存最终答案或一个批次总 Token，再用字符数估算费用，缺少单次请求的模型、输入/输出 Token、延迟、停止原因和归属标识。这会把重试、重复请求和高价模型路由隐藏在总数里；缺失定价时若又把成本记为零，监控看似正常，但超支只能在账单到达后被动发现，也无法追责或优化。

**解决方案**：本 Notebook 将实现一个极简的 Cost Telemetry，采用`Provider 原始 usage + 逐请求成本事件 + Trace 关联 + 版本化定价 + 分层预算告警`机制：优先记录 provider 返回的真实 usage，为每次请求绑定 task、step、model 和 trace ID，依据带版本的价格表换算费用，并在单步与整个任务级别检查预算。这与 OpenTelemetry GenAI 语义约定和 FinOps 成本分摊思路对齐，并为 FrugalGPT 式级联路由、缓存与限额策略提供可验证的观测依据。然后在同一任务和同一真实 API 响应上对比：基线版本只有模糊总数，无法找到超支步骤；改进版本生成可回溯账册并准确触发预算告警，从而直观看到成本遥测如何把“花了多少”变成“谁在哪一步为什么花了多少”。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 定义模型提交格式
后续要比较两次调用的成本，因此先让模型用同一种结构提交日志等级。`step_id` 表示结果属于哪一步，`level` 只允许填写 `low` 或 `high`，这样任务答案不会与成本记录混在一起。

In [2]:
# step_id 把模型答案关联到具体步骤
# level 使用固定选项，便于比较两次调用的任务结果
tools = [{
    "type": "function",
    "function": {
        "name": "submit_incident_level",
        "description": "提交当前步骤的日志等级",
        "parameters": {
            "type": "object",
            "properties": {
                "step_id": {"type": "string"},
                "level": {"type": "string", "enum": ["low", "high"]},
            },
            "required": ["step_id", "level"],
        },
    },
}]

print("模型提交工具：", tools[0]["function"]["name"])
print("必填字段：", tools[0]["function"]["parameters"]["required"])

模型提交工具： submit_incident_level
必填字段： ['step_id', 'level']


输出显示模型只能通过 `submit_incident_level` 提交步骤编号和日志等级。这个工具只固定答案格式，不记录 Token；下一步准备两份长度明显不同的输入。

## 2.2 准备两份不同长度的输入
生产系统常把检索结果、历史消息或重复日志全部塞进请求，真正的费用可能花在输入上下文，而不是最终答案。下面准备两个相同类型的分级任务：第一份只有一句正常日志，第二份包含大量重复日志和一条关键故障。

In [3]:
# 短输入只包含判断所需的事实
# 长输入重复无关日志，用来复现生产中的上下文膨胀
short_context = "服务运行正常，没有错误。"
repeated_log = "例行日志：健康检查通过。"
long_context = (repeated_log + "\n") * 40 + "关键日志：支付请求连续失败。"

requests = [
    {"step_id": "short_context", "context": short_context},
    {"step_id": "long_context", "context": long_context},
]

for item in requests:
    print(item["step_id"], "字符数：", len(item["context"]))

short_context 字符数： 12
long_context 字符数： 534


输出显示两次调用处理同一种任务，但 `long_context` 的输入明显更长。此时还没有调用模型，也不能用字符数冒充 Token；下一步只固定统一的逐步骤 Token 预算。

## 2.3 固定逐步骤预算
预算必须在请求发出前确定，否则看到结果后再改阈值就失去了约束意义。下面规定每一步最多使用 300 Token，后续是否越线只看 provider 返回的真实总 Token。

In [4]:
# 预算同时包含输入 Token 和输出 Token
# 两条执行路径始终使用同一个固定阈值
step_token_budget = 300

print("每步 Token 预算：", step_token_budget)

每步 Token 预算： 300


输出说明每次模型调用都使用 300 Token 的同一预算。现在只确定了上限，还不知道实际消耗；下一步固定任务答案和成本账册共同满足的成功标准。

## 2.4 定义成功标准
只把日志等级答对还不够，Cost Telemetry 还必须为每一步留下可以回溯的真实用量。下面固定正确等级和账册必需字段，后面的基线版本与改进版本都使用这同一把尺子。

In [5]:
# expected_levels 固定两次日志分级的正确答案
# required_ledger_fields 固定每一步必须留下的成本证据
expected_levels = {
    "short_context": "low",
    "long_context": "high",
}
required_ledger_fields = [
    "step_id",
    "model",
    "input_tokens",
    "output_tokens",
    "total_tokens",
    "latency_ms",
    "stop_reason",
    "over_budget",
]

print("正确等级：", expected_levels)
print("账册字段：", required_ledger_fields)

正确等级： {'short_context': 'low', 'long_context': 'high'}
账册字段： ['step_id', 'model', 'input_tokens', 'output_tokens', 'total_tokens', 'latency_ms', 'stop_reason', 'over_budget']


输出给出了唯一成功标准：两次分级都要正确，并且两次调用都要留下与真实响应一致的逐步骤账册和预算状态。至此，提交格式、输入、预算和标准都已固定；下一章将发送两次真实 API 请求并保存原始结果。

# 3. 获取并验证 API 响应
## 3.1 发送两次真实 API 请求
现在把第 2 章固定的短、长输入依次交给 `.env` 指定的同一模型。两次请求使用相同指令和相同工具，只改变日志上下文，并分别保存原始响应与实际等待时间。

In [6]:
from time import perf_counter

# responses 保存 provider 返回的完整原始响应
# api_latency_ms 保存每一步真实等待时间
responses = {}
api_latency_ms = {}

for item in requests:
    messages = [
        {
            "role": "system",
            "content": "服务正常且没有错误时为 low；出现支付请求连续失败时为 high。必须调用工具，并原样返回 step_id。",
        },
        {
            "role": "user",
            "content": f"step_id：{item['step_id']}\n日志：\n{item['context']}",
        },
    ]

    request_started = perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        tools=tools,
        tool_choice="required",
        temperature=0,
    )
    elapsed_ms = round((perf_counter() - request_started) * 1000)

    responses[item["step_id"]] = response
    api_latency_ms[item["step_id"]] = elapsed_ms
    print(item["step_id"], "真实 API 响应已收到")

short_context 真实 API 响应已收到


long_context 真实 API 响应已收到


输出说明两次请求都由真实 API 返回，完整响应分别保存在 `responses` 中。此时还没有读取模型决定，也没有计算任何成本；下一步取出两次结构化工具参数。

## 3.2 读取模型决定
模型的最终选择保存在工具参数中，而不是普通文字里。下面逐个还原 JSON 参数，并按照请求时的步骤编号保存，避免把任务答案与 Token 用量混在一起。

In [7]:
import json

# tool_calls 保存模型提交的结构化决定
# model_results 按原请求步骤保存解析后的参数
model_results = {}

for item in requests:
    response = responses[item["step_id"]]
    tool_call = response.choices[0].message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    model_results[item["step_id"]] = arguments
    print(item["step_id"], "模型决定：", arguments)

short_context 模型决定： {'step_id': 'short_context', 'level': 'low'}
long_context 模型决定： {'step_id': 'long_context', 'level': 'high'}


输出展示了两次真实模型决定：短输入应被判为 `low`，包含支付连续失败的长输入应被判为 `high`。下一步使用第 2 章固定的答案判断两次调用是否完成了日志分级任务。

## 3.3 判断任务答案
成本对照必须建立在任务本身正确的前提上。下面同时检查模型返回的步骤编号和日志等级，确保后续差异只来自外层 Cost Telemetry，而不是模型答错。

In [8]:
# 每一步都要返回原步骤编号和对应的正确等级
# task_success 只有在两步全部正确时才为 True
task_checks = {}

for step_id in expected_levels:
    result = model_results[step_id]
    id_correct = result["step_id"] == step_id
    level_correct = result["level"] == expected_levels[step_id]
    task_checks[step_id] = id_correct and level_correct

task_success = all(task_checks.values())
print("逐步结果：", task_checks)
print("任务成功：", task_success)

逐步结果： {'short_context': True, 'long_context': True}
任务成功： True


输出中的两项检查和总结果都为 `True`，说明真实模型正确完成了两次日志分级。任务答案已经固定，下一步只查看 provider 随原始响应返回的运行信息。

## 3.4 查看原始请求信息
每个真实响应都带有输入 Token、输出 Token、总 Token 和停止原因，等待时间则来自刚才的实际计时。下面直接展示这些原始事实，但暂时不把它们加工成成本账册。

In [9]:
# Token 数直接读取 provider usage，不使用字符数估算
# stop_reason 只表示 API 如何结束，不代表完整任务是否成功
for item in requests:
    step_id = item["step_id"]
    response = responses[step_id]
    usage = response.usage
    raw_info = {
        "model": model_name,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "latency_ms": api_latency_ms[step_id],
        "stop_reason": response.choices[0].finish_reason,
    }
    print(step_id, "原始信息：", raw_info)

short_context 原始信息： {'model': 'LongCat-2.0', 'input_tokens': 206, 'output_tokens': 126, 'total_tokens': 332, 'latency_ms': 3892, 'stop_reason': 'tool_calls'}
long_context 原始信息： {'model': 'LongCat-2.0', 'input_tokens': 527, 'output_tokens': 58, 'total_tokens': 585, 'latency_ms': 3135, 'stop_reason': 'tool_calls'}


输出记录了同一模型两次真实调用的 Token、延迟和停止原因。长上下文带来了更高的输入用量，但这些事实仍分散在各自响应里；下一章将定义生产中常见的字符估算基线，观察它为何看不见真正的输入成本。

# 4. 定义基线组件
生产中常见的错误基线是拿到模型结果后，只按“4 个字符约等于 1 Token”估算用量。这种方法实现简单，却只看得见输出文字，看不见发送给模型的输入上下文，也没有使用 provider 已经返回的真实 Token。下面原样复现这个基线。

In [10]:
# 基线只读取模型提交的工具参数文本
# 四字符公式忽略输入上下文和真实分词结果
def estimate_tokens_from_output(response):
    arguments = response.choices[0].message.tool_calls[0].function.arguments
    estimated_tokens = len(arguments) // 4
    return estimated_tokens

print("基线组件：按输出字符数估算 Token")
print("估算公式：字符数 // 4")

基线组件：按输出字符数估算 Token
估算公式：字符数 // 4


输出说明字符估算基线已经定义，但尚未处理第 3 章的真实响应。下一章会把短、长两次响应交给同一个函数，观察只看输出字符能否发现输入上下文带来的预算超支。

# 5. 展示基线故障
## 5.1 运行字符估算基线
现在把第 3 章的两次真实响应交给第 4 章函数。基线只根据模型提交的工具参数估算 Token，再用第 2 章固定的 300 Token 上限判断是否需要告警。

In [11]:
# baseline_rows 保存字符公式得到的逐步估算
# baseline_alerts 只收集估算值超过预算的步骤
baseline_rows = []
baseline_alerts = []

for item in requests:
    step_id = item["step_id"]
    estimated_tokens = estimate_tokens_from_output(responses[step_id])
    over_budget = estimated_tokens > step_token_budget
    row = {
        "step_id": step_id,
        "estimated_tokens": estimated_tokens,
        "over_budget": over_budget,
    }
    baseline_rows.append(row)

    if over_budget:
        baseline_alerts.append(step_id)

    print(row)

{'step_id': 'short_context', 'estimated_tokens': 11, 'over_budget': False}
{'step_id': 'long_context', 'estimated_tokens': 11, 'over_budget': False}


输出显示两个估算值都远低于 300，基线因此认为短、长两步都没有超预算。它只看见长度相近的工具参数，完全没有计入两份不同长度的输入上下文；下一步用真实 usage 检查这个判断。

## 5.2 判断基线结果
评分时只把 provider 返回的真实总 Token 当作事实，并继续使用第 2 章规定的账册字段。下面比较基线告警与真实超预算步骤，同时检查基线记录是否足以回溯每次调用。

In [12]:
# actual_alerts 根据 provider 的真实总 Token 生成
# ledger_complete 检查基线是否包含统一要求的全部字段
actual_alerts = []

for item in requests:
    step_id = item["step_id"]
    actual_tokens = responses[step_id].usage.total_tokens

    if actual_tokens > step_token_budget:
        actual_alerts.append(step_id)

baseline_fields = set(baseline_rows[0])
ledger_complete = set(required_ledger_fields).issubset(baseline_fields)
alert_correct = baseline_alerts == actual_alerts
baseline_success = task_success and ledger_complete and alert_correct

print("基线告警：", baseline_alerts)
print("真实超预算步骤：", actual_alerts)
print("账册字段完整：", ledger_complete)
print("基线成功：", baseline_success)

基线告警： []
真实超预算步骤： ['short_context', 'long_context']
账册字段完整： False
基线成功： False


输出中的基线告警为空，但真实超预算列表至少包含 `long_context`；账册字段也不完整，因此基线结果为 `False`。模型已经正确完成任务，失败来自外层程序用输出字符猜测成本，既漏掉输入 Token，也无法留下可回溯的调用事实。下一章将定义直接读取 provider usage 的改进组件。

# 6. 定义改进组件
截至 2026 年，OpenTelemetry GenAI 语义约定与 FinOps 成本分摊共同强调同一条主线：每次模型调用都应产生一条可关联、可聚合的成本事件，Token 以 provider usage 为事实源，再附上步骤、模型、延迟、停止原因和预算状态。下面用一个函数实现这个最小边界，不再根据字符数猜测。

In [13]:
# usage 提供本次请求真实的输入、输出和总 Token
# step_id 把成本事件关联到产生费用的具体步骤
def create_cost_event(step_id, response, latency_ms, token_budget):
    usage = response.usage

    return {
        "step_id": step_id,
        "model": response.model,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "latency_ms": latency_ms,
        "stop_reason": response.choices[0].finish_reason,
        "over_budget": usage.total_tokens > token_budget,
    }

print("改进组件：provider usage -> 逐步骤成本事件")
print("事件字段：", required_ledger_fields)

改进组件：provider usage -> 逐步骤成本事件
事件字段： ['step_id', 'model', 'input_tokens', 'output_tokens', 'total_tokens', 'latency_ms', 'stop_reason', 'over_budget']


输出说明改进组件已经定义，生成的字段与第 2 章成功标准完全一致，但此时还没有处理任何响应。下一章会把第 3 章同一批真实响应转换成逐步骤账册，并查看预算状态。

# 7. 展示修复结果
## 7.1 生成逐步骤成本账册
现在把第 3 章保存的同一批真实响应交给第 6 章组件。每个响应生成一条成本事件，模型答案和 API 调用都不改变，唯一变化是外层程序开始读取并关联真实 usage。

In [14]:
# cost_ledger 按请求顺序保存逐步骤成本事件
# 每条事件都复用本次 rollout 的真实响应和延迟
cost_ledger = []

for item in requests:
    step_id = item["step_id"]
    event = create_cost_event(
        step_id,
        responses[step_id],
        api_latency_ms[step_id],
        step_token_budget,
    )
    cost_ledger.append(event)
    print(event)

{'step_id': 'short_context', 'model': 'LongCat-2.0', 'input_tokens': 206, 'output_tokens': 126, 'total_tokens': 332, 'latency_ms': 3892, 'stop_reason': 'tool_calls', 'over_budget': True}
{'step_id': 'long_context', 'model': 'LongCat-2.0', 'input_tokens': 527, 'output_tokens': 58, 'total_tokens': 585, 'latency_ms': 3135, 'stop_reason': 'tool_calls', 'over_budget': True}


输出中的每条事件都明确记录了费用属于哪一步，并保留真实输入 Token、输出 Token、总 Token、延迟与停止原因。`over_budget` 直接比较真实总量和 300 Token 上限；下一步把超预算事件提取成告警列表。

## 7.2 生成预算告警
账册负责保存事实，告警只负责指出哪些步骤已经越线。下面逐条读取 `over_budget`，保留超预算步骤编号，方便直接定位费用来源。

In [15]:
# cost_alerts 只保存超过固定预算的步骤
# 告警依据来自账册中的真实 provider usage
cost_alerts = []

for event in cost_ledger:
    if event["over_budget"]:
        cost_alerts.append(event["step_id"])

print("成本告警：", cost_alerts)

成本告警： ['short_context', 'long_context']


输出直接列出了本次真实运行超过 300 Token 的步骤，不再把长输入隐藏在输出字符估算中。具体列表取决于本次真实 usage，但一定能从账册回到对应调用；下一步使用统一标准判断修复结果。

## 7.3 判断修复结果
最后仍使用第 2 章的同一把尺子：模型任务必须正确，账册字段必须完整，Token 与运行信息必须来自原响应，告警也必须覆盖真实超预算步骤。

In [16]:
# ledger_complete 检查每条事件是否满足统一字段合同
# runtime_exact 检查事件是否忠实记录原始 API 事实
ledger_complete = True
runtime_exact = True

for event in cost_ledger:
    if set(event) != set(required_ledger_fields):
        ledger_complete = False

    step_id = event["step_id"]
    response = responses[step_id]
    usage = response.usage
    tokens_exact = (
        event["input_tokens"] == usage.prompt_tokens
        and event["output_tokens"] == usage.completion_tokens
        and event["total_tokens"] == usage.total_tokens
    )
    call_exact = (
        event["latency_ms"] == api_latency_ms[step_id]
        and event["stop_reason"] == response.choices[0].finish_reason
    )

    if not tokens_exact or not call_exact:
        runtime_exact = False

alerts_exact = cost_alerts == actual_alerts
fixed_success = task_success and ledger_complete and runtime_exact and alerts_exact

print("账册字段完整：", ledger_complete)
print("运行信息准确：", runtime_exact)
print("预算告警准确：", alerts_exact)
print("修复成功：", fixed_success)

账册字段完整： True
运行信息准确： True
预算告警准确： True
修复成功： True


输出中的四项结果全部为 `True`。模型、任务和真实响应没有变化，修复只发生在外层 Cost Telemetry：字符猜测被逐请求 provider usage 取代，每笔用量都有步骤归属，预算告警也能准确定位。下一章将汇总基线版本与改进版本的消融对照。

# 8. 汇总消融对照
## 8.1 整理两条路径的同源数据
基线版本和改进版本复用同一模型、同一任务与同两次 API 响应，因此真实 Token、延迟和调用次数完全相同。下面只汇总外层程序如何记录这些事实，以及最终是否满足统一成功标准。

In [17]:
# 真实总量只从第 3 章的 provider 响应累计
# 两个版本共享这些调用事实，避免把模型差异混入消融
actual_total_tokens = 0
actual_total_latency_ms = 0
baseline_reported_tokens = 0

for item in requests:
    step_id = item["step_id"]
    actual_total_tokens += responses[step_id].usage.total_tokens
    actual_total_latency_ms += api_latency_ms[step_id]

for row in baseline_rows:
    baseline_reported_tokens += row["estimated_tokens"]

baseline_ledger_complete = set(required_ledger_fields).issubset(set(baseline_rows[0]))
fixed_ledger_complete = set(cost_ledger[0]) == set(required_ledger_fields)

ablation_rows = [
    {
        "版本": "字符估算基线",
        "Token 来源": "输出字符",
        "报告 Token": baseline_reported_tokens,
        "真实 Token": actual_total_tokens,
        "真实延迟(ms)": actual_total_latency_ms,
        "API 次数": len(responses),
        "账册完整": baseline_ledger_complete,
        "预算告警": baseline_alerts,
        "任务正确": task_success,
        "Harness 成功": baseline_success,
    },
    {
        "版本": "Cost Telemetry",
        "Token 来源": "provider usage",
        "报告 Token": actual_total_tokens,
        "真实 Token": actual_total_tokens,
        "真实延迟(ms)": actual_total_latency_ms,
        "API 次数": len(responses),
        "账册完整": fixed_ledger_complete,
        "预算告警": cost_alerts,
        "任务正确": task_success,
        "Harness 成功": fixed_success,
    },
]

print("对照版本数：", len(ablation_rows))
print("共享真实 API 次数：", len(responses))

对照版本数： 2
共享真实 API 次数： 2


输出说明两条对照记录已经准备好，并共同使用两次真实 API 调用。下一步把所有字段放到同一张纯文本表中，直接比较报告值、真实值、账册、告警和任务结果。

## 8.2 展示完整对照
下面按照固定列顺序打印两条记录。这样既能看到模型任务是否正确，也能看到 Cost Telemetry 是否改变了真实 API 消耗，避免把“看清成本”误写成“已经降低成本”。

In [18]:
# columns 固定表格字段顺序，便于逐项横向比较
# 每一行都来自上一格已经整理好的同源数据
columns = list(ablation_rows[0])
print(" | ".join(columns))

for row in ablation_rows:
    values = []

    for column in columns:
        values.append(str(row[column]))

    print(" | ".join(values))

版本 | Token 来源 | 报告 Token | 真实 Token | 真实延迟(ms) | API 次数 | 账册完整 | 预算告警 | 任务正确 | Harness 成功
字符估算基线 | 输出字符 | 22 | 917 | 7027 | 2 | False | [] | True | False
Cost Telemetry | provider usage | 917 | 917 | 7027 | 2 | True | ['short_context', 'long_context'] | True | True


表格显示两条路径的 API 次数、真实 Token、真实延迟和任务正确性完全相同。字符基线只报告很小的估算值，没有完整账册和预算告警；Cost Telemetry 报告值与真实值一致，并使 Harness 从失败变为成功。下一步用状态变化收束因果关系。

## 8.3 总结机制效果
最后只保留最能说明因果关系的五项变化。真实 API 用量保持不变，说明本实验测到的是可观测性改善，而不是路由、缓存或更换模型带来的成本优化。

In [19]:
# 左侧是字符估算基线，右侧是 Cost Telemetry
# 真实调用保持不变，只有外层记录与告警机制发生变化
telemetry_effect = {
    "任务正确": f"{task_success} -> {task_success}",
    "真实 Token": f"{actual_total_tokens} -> {actual_total_tokens}",
    "账册完整": f"{baseline_ledger_complete} -> {fixed_ledger_complete}",
    "预算告警": f"{baseline_alerts} -> {cost_alerts}",
    "Harness 成功": f"{baseline_success} -> {fixed_success}",
}

for name, change in telemetry_effect.items():
    print(name, "：", change)

任务正确 ： True -> True
真实 Token ： 917 -> 917
账册完整 ： False -> True
预算告警 ： [] -> ['short_context', 'long_context']
Harness 成功 ： False -> True


输出中的任务正确性和真实 Token 都没有变化，账册却从不完整变为完整，预算告警从空列表变为真实超预算步骤，Harness 最终从 `False` 变为 `True`。这说明模型一直能完成任务，真正的故障是外层程序没有使用 provider usage，也无法把消耗归到具体步骤。

## 8.4 拓展

### nano 版省略了什么

nano 版只汇总单次运行的 Token、调用次数与延迟，没有供应商账单对账、缓存 Token、推理 Token、工具基础设施成本、价格版本、货币、租户配额、p95/p99 和预算预测。生产 Cost Telemetry 必须保留未知值而不是写零，并将每笔用量归因到 run、span、模型和业务结果。

### 延伸阅读


1. 2026, [OpenTelemetry, GenAI semantic conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/)：GenAI 请求、Token、事件与 Agent Span 的跨平台字段约定。
2. 2026, [OpenLLMetry](https://github.com/traceloop/openllmetry)：基于 OpenTelemetry 的主流 LLM/Agent 自动埋点。
3. 2025, [Beyond Black-Box Benchmarking](https://arxiv.org/abs/2503.06745)：用运行遥测解释 Agent 质量、成本与性能。